# Load Data

In [1]:
import os
import glob

data_path = "../Qwen-Audio/data/情绪标柱/cut_videos"

audio_paths = glob.glob(os.path.join(data_path, "*.mp4"))
audio_paths

['../Qwen-Audio/data/情绪标柱/cut_videos/厌恶情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/开心情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/悲伤情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/中性情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/愤怒情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/害怕情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/警惕情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/惊讶情绪_cut.mp4',
 '../Qwen-Audio/data/情绪标柱/cut_videos/赞赏情绪_cut.mp4']

In [2]:
from io import BytesIO
from urllib.request import urlopen
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

model_path = "/home/ec2-user/SageMaker/efs/Models/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)
model = Qwen2AudioForConditionalGeneration.from_pretrained(model_path, device_map="auto")

conversation = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"},
    ]},
    {"role": "assistant", "content": "Yes, the speaker is female and in her twenties."},
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/translate_to_chinese.wav"},
    ]},
]
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios = []
for message in conversation:
    if isinstance(message["content"], list):
        for ele in message["content"]:
            if ele["type"] == "audio":
                audios.append(librosa.load(
                    BytesIO(urlopen(ele['audio_url']).read()), 
                    sr=processor.feature_extractor.sampling_rate)[0]
                )

inputs = processor(text=text, audios=audios, return_tensors="pt", padding=True)
inputs.input_ids = inputs.input_ids.to("cuda")
generate_ids = model.generate(**inputs, max_length=2048)
generate_ids = generate_ids[:, inputs.input_ids.size(1):]

response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print(f"response: {response}")


/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 5/5 [00:44<00:00,  8.98s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `i

response: 每个人都希望被欣赏，所以如果你欣赏某人，不要把它保密。


In [3]:




# from transformers import AutoModelForCausalLM, AutoTokenizer
# from transformers.generation import GenerationConfig
# import torch
# torch.manual_seed(1234)


# # Note: The default behavior now has injection attack prevention off.
# tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# # use bf16
# # model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-Audio-Chat", device_map="auto", trust_remote_code=True, bf16=True).eval()
# # use fp16
# # model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-Audio-Chat", device_map="auto", trust_remote_code=True, fp16=True).eval()
# # use cpu only
# # model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-Audio-Chat", device_map="cpu", trust_remote_code=True).eval()
# # use cuda device
# model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda", trust_remote_code=True).eval()

# # Specify hyperparameters for generation (No need to do this if you are using transformers>4.32.0)
# # model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-Audio-Chat", trust_remote_code=True)

# # 1st dialogue turn
# query = tokenizer.from_list_format([
#     {'audio': 'assets/audio/1272-128104-0000.flac'}, # Either a local path or an url
#     {'text': 'what does the person say?'},
# ])
# response, history = model.chat(tokenizer, query=query, history=None)
# print(response)
# # The person says: "mister quilter is the apostle of the middle classes and we are glad to welcome his gospel".

# # 2nd dialogue turn
# response, history = model.chat(tokenizer, 'Find the start time and end time of the word "middle classes"', history=history)
# print(response)
# The word "middle classes" starts at <|2.33|> seconds and ends at <|3.26|> seconds.

In [6]:
def invoke_model(audio_path):
    prompt = """\
请根据音频中的人物声音及说话内容，从["中性", "害怕", "厌恶", "开心", "悲伤", "惊讶", "愤怒", "警惕", "赞赏"]这9个情感类别标签中选择最符合的类别标签。只输出音频的情感标签，不要输出任何其它信息。
"""
    conversation = [
        {'role': 'system', 'content': 'You are a helpful assistant.'}, 
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios = []
    for message in conversation:
        if isinstance(message["content"], list):
            for ele in message["content"]:
                if ele["type"] == "audio":
                    if ele['audio_url'].startswith(('http://', 'https://')):
                        # Handle URL
                        audio_data = librosa.load(
                            BytesIO(urlopen(ele['audio_url']).read()), 
                            sr=processor.feature_extractor.sampling_rate)[0]
                    else:
                        # Handle local file path
                        audio_data = librosa.load(
                            ele['audio_url'], 
                            sr=processor.feature_extractor.sampling_rate)[0]
                    
                    audios.append(audio_data)

    inputs = processor(text=text, audios=audios, return_tensors="pt", padding=True)
    inputs.input_ids = inputs.input_ids.to("cuda")
    generate_ids = model.generate(**inputs, max_length=2048)
    generate_ids = generate_ids[:, inputs.input_ids.size(1):]

    response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    return response

In [7]:
from tqdm import tqdm
import pandas as pd

result = []
for audio_path in tqdm(audio_paths, total=len(audio_paths)):
    response = invoke_model(audio_path)
    print(response)
    result.append([audio_path, response])

df = pd.DataFrame(result, columns=['audio', 'emotion'])
df.head(10)

  0%|          | 0/9 [00:00<?, ?it/s]/tmp/ipykernel_40489/1497556652.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data = librosa.load(
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Pl

{'情绪': '愤怒'}


/tmp/ipykernel_40489/1497556652.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data = librosa.load(
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `inp

开心


/tmp/ipykernel_40489/1497556652.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data = librosa.load(
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `inp

...

/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
 89%|████████▉ | 8/9 [01:30<00:16, 16.64s/it]

{'情绪': '惊讶', '文本': \"这段音频中人物的情绪是带有惊讶的语气。\"}


/tmp/ipykernel_40489/1497556652.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data = librosa.load(
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.
/home/ec2-user/SageMaker/efs/conda_envs/qwen_audio/lib/python3.10/site-packages/transformers/generation/utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `inp

赞赏


,audio,emotion
0,../Qwen-Audio/data/情绪标柱/cut_videos/厌恶情绪_cut.mp4,{'情绪': '愤怒'}
1,../Qwen-Audio/data/情绪标柱/cut_videos/开心情绪_cut.mp4,开心
2,../Qwen-Audio/data/情绪标柱/cut_videos/悲伤情绪_cut.mp4,悲伤
3,../Qwen-Audio/data/情绪标柱/cut_videos/中性情绪_cut.mp4,中性
4,../Qwen-Audio/data/情绪标柱/cut_videos/愤怒情绪_cut.mp4,{'情绪': '愤怒'}
5,../Qwen-Audio/data/情绪标柱/cut_videos/害怕情绪_cut.mp4,{'情绪': '悲伤'}
6,../Qwen-Audio/data/情绪标柱/cut_videos/警惕情绪_cut.mp4,警惕
7,../Qwen-Audio/data/情绪标柱/cut_videos/惊讶情绪_cut.mp4,"{'情绪': '惊讶', '文本': \""这段音频中人物的情绪是带有惊讶的语气。\""}"
8,../Qwen-Audio/data/情绪标柱/cut_videos/赞赏情绪_cut.mp4,赞赏


In [9]:
df.to_excel("outputs/qwen2_audio_eval.xlsx", index=False)